<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/Topic_Prabowo%20ke%20Vladivostok_top_author_sentimen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Install Dependency

In [1]:
!pip install -q -U google-genai pandas tqdm emoji

## 1. Import Library

In [2]:
import os
import re
import json
import time
import hashlib
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from google import genai
from google.genai import types
from google.colab import drive

try:
    import emoji
    HAS_EMOJI_LIB = True
except ImportError:
    HAS_EMOJI_LIB = False
    print("[WARN] library 'emoji' tidak ada -> emoji akan dibuang, bukan dikonversi jadi teks.")

print("Library siap.")

Library siap.


## 2. Set API Key Anthropic

Key diketik lewat input tersembunyi (tidak ke-log di notebook), jadi aman
kalau notebook ini nanti di-upload ke GitHub.

Alternatif lebih nyaman: pakai fitur **Secrets** Colab (ikon kunci di sidebar
kiri), simpan sebagai `ANTHROPIC_API_KEY`, lalu ganti isi cell ini dengan:

```python
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
```


In [3]:
import os
from google.colab import userdata

# Gemini (dipakai untuk klasifikasi tema & NER di step-step sebelumnya)
os.environ["GOOOGLE_API_KEY"] = userdata.get("GOOOGLE_API_KEY")

print("API key sudah di-set.")

API key sudah di-set.


## 3. Konfigurasi

Edit bagian ini kalau nama kolom, daftar tema, atau bobot ranking berubah.
Semua cell di bawah memakai variabel dari sini.


In [4]:
# ---------------------------------------------------------------
# Step 1 - Load Data (otomatis cek: sudah ada hasil klasifikasi tema sebelumnya?
# kalau ada -> load pickle langsung (skip download & Step 3).
# kalau belum -> download & load .xlsx dari Google Drive seperti biasa)
# ---------------------------------------------------------------
drive.mount('/content/drive')

PICKLE_PATH_AFTER_THEME = "/content/drive/MyDrive/df_clean_after_theme.pkl"
RESUME_FROM_THEME = os.path.exists(PICKLE_PATH_AFTER_THEME)

if RESUME_FROM_THEME:
    # Jalur A: sudah pernah selesai klasifikasi tema -> load langsung
    df_clean = pd.read_pickle(PICKLE_PATH_AFTER_THEME)
    print(f"[INFO] Ditemukan hasil klasifikasi tema sebelumnya di {PICKLE_PATH_AFTER_THEME}")
    print(f"[INFO] df_clean dimuat langsung, {len(df_clean)} baris. Step 1b & Step 3 BISA DILEWATI.")
    print(df_clean["theme"].value_counts())

else:
    # Jalur B: belum ada hasil sebelumnya -> install gdown & download xlsx seperti biasa
    !pip install -q gdown openpyxl
    import gdown

    DRIVE_XLSX_URL = "https://docs.google.com/spreadsheets/d/1RWje0fuPtl1jHVzibr480gWfK0ivJkbY/edit?usp=sharing&ouid=116825097454650626545&rtpof=true&sd=true"
    SHEET_NAME = "Sheet1"

    def extract_drive_file_id(url: str) -> str:
        match = re.search(r"/d/([a-zA-Z0-9_-]+)", url)
        if match:
            return match.group(1)
        match = re.search(r"[?&]id=([a-zA-Z0-9_-]+)", url)
        if match:
            return match.group(1)
        raise ValueError("Tidak bisa menemukan file ID dari URL. Pastikan format link Google Drive benar.")

    file_id = extract_drive_file_id(DRIVE_XLSX_URL)
    local_path = "data_input.xlsx"

    print(f"[INFO] Tidak ditemukan hasil sebelumnya, mulai dari awal.")
    print(f"[INFO] File ID terdeteksi: {file_id}")
    print("Mengunduh file .xlsx dari Google Drive...")
    gdown.download(f"https://drive.google.com/uc?id={file_id}", local_path, quiet=False)

    xls = pd.ExcelFile(local_path)
    print(f"\n[INFO] Sheet yang tersedia di file ini: {xls.sheet_names}")

    df_raw = pd.read_excel(local_path, sheet_name=SHEET_NAME, skiprows=1)

    print(f"\n[OK] {len(df_raw)} baris, {len(df_raw.columns)} kolom berhasil dimuat")
    print("Kolom:", list(df_raw.columns))
    display(df_raw.head(5))

# ---------------------------------------------------------------
# Mapping nama kolom sesuai header asli di spreadsheet
# (No, Type, Headline, Mentions, Date, Link, Media, Sentiment, Author, Followers, Retweeted, Favourited)
# Ini SELALU didefinisikan di kedua jalur, karena dipakai di step-step berikutnya
# (baik Step 1b/2/3 yang butuh df_raw, maupun Step NER/top-author yang cuma butuh df_clean)
# ---------------------------------------------------------------
COL_NO = "No"
COL_TYPE = "Type"
COL_HEADLINE = "Headline"
COL_MENTIONS = "Mentions"
COL_DATE = "Date"
COL_LINK = "Link"
COL_MEDIA = "Media"
COL_SENTIMENT = "Sentiment"
COL_AUTHOR_ID = "Author"
COL_FOLLOWERS = "Followers"
COL_RETWEETED = "Retweeted"
COL_FAVOURITED = "Favourited"

# ---------------------------------------------------------------
# Daftar tema
# ---------------------------------------------------------------
THEMES = ["kunjungan_prabowo", "rusia_putin", "lainnya"]
THEME_DESCRIPTIONS = {
    "kunjungan_prabowo": (
        "kunjungan Prabowo ke Rusia, kunjungan kenegaraan Prabowo, lawatan Prabowo, agenda Prabowo di Rusia, "
        "delegasi Indonesia ke Rusia, Prabowo temui Putin, pertemuan bilateral Prabowo Putin, "
        "Prabowo di Vladivostok, Prabowo di Moskow, Prabowo hadiri Eastern Economic Forum, "
        "Prabowo EEF, Prabowo forum ekonomi Rusia, kunjungan presiden RI ke Rusia, "
        "kerja sama Indonesia Rusia, nota kesepahaman Indonesia Rusia, MoU Indonesia Rusia, "
        "investasi Rusia ke Indonesia, kunjungan balasan Prabowo, protokoler kunjungan Prabowo, "
        "rombongan menteri dampingi Prabowo, jadwal kunjungan Prabowo Rusia, "
        "hasil pertemuan Prabowo Putin, pernyataan bersama Indonesia Rusia, "
        "kunjungan kenegaraan ke Kremlin, Prabowo di Kremlin, sambutan kenegaraan Prabowo Rusia, "
        "kereta kepresidenan Rusia, upacara penyambutan Prabowo, kunjungan luar negeri Prabowo, "
        "diplomasi Prabowo Rusia, agenda strategis Indonesia Rusia, "
        "lawatan kenegaraan Presiden Prabowo, kunjungan resmi Presiden Prabowo ke Rusia, "
        "Prabowo Subianto ke Rusia, Prabowo terbang ke Rusia, keberangkatan Prabowo ke Rusia, "
        "Prabowo tiba di Rusia, Prabowo pulang dari Rusia, kepulangan Prabowo dari Rusia, "
        "menteri luar negeri dampingi Prabowo, Sugiono dampingi Prabowo, Menlu RI ke Rusia, "
        "menteri pertahanan dampingi Prabowo, delegasi bisnis Indonesia Rusia, "
        "pengusaha Indonesia ikut kunjungan Rusia, pebisnis dampingi Prabowo, "
        "kesepakatan dagang Indonesia Rusia, perjanjian kerja sama Indonesia Rusia, "
        "kontrak dagang Indonesia Rusia, ekspor impor Indonesia Rusia, "
        "kerja sama pertahanan Indonesia Rusia, alutsista Rusia untuk Indonesia, "
        "kerja sama energi Indonesia Rusia, kerja sama nuklir Indonesia Rusia, "
        "PLTN Rusia Indonesia, Rosatom Indonesia, kerja sama pangan Indonesia Rusia, "
        "kunjungan Prabowo pasca kunjungan ke China, kunjungan Prabowo pasca KTT, "
        "reaksi publik kunjungan Prabowo Rusia, kritik kunjungan Prabowo ke Rusia, "
        "pujian kunjungan Prabowo ke Rusia, kontroversi kunjungan Prabowo Rusia, "
        "netralitas Indonesia kunjungan Rusia, politik luar negeri bebas aktif Prabowo, "
        "sikap Barat soal kunjungan Prabowo Rusia, respons AS soal kunjungan Prabowo Rusia, "
        "istana kepresidenan soal kunjungan Rusia, juru bicara presiden soal kunjungan Rusia, "
        "foto kunjungan Prabowo Rusia, video kunjungan Prabowo Rusia, momen Prabowo di Rusia, "
        "red carpet Prabowo Rusia, karpet merah Prabowo Rusia, penyambutan militer Prabowo Rusia"
    ),
    "rusia_putin": (
        "Rusia, Vladimir Putin, Presiden Rusia, Kremlin, Vladivostok, Moskow, Rusia Timur Jauh, "
        "Eastern Economic Forum, EEF Vladivostok, forum ekonomi Rusia, kebijakan luar negeri Rusia, "
        "hubungan Rusia dengan negara lain, sanksi terhadap Rusia, sanksi Barat ke Rusia, "
        "ekonomi Rusia, perdagangan Rusia, energi Rusia, gas Rusia, minyak Rusia, "
        "militer Rusia, angkatan bersenjata Rusia, perang Rusia Ukraina, konflik Rusia Ukraina, "
        "geopolitik Rusia, pernyataan Putin, pidato Putin, kebijakan Putin, "
        "Kementerian Luar Negeri Rusia, duta besar Rusia, kedutaan Rusia, "
        "kerja sama BRICS, Rusia BRICS, aliansi Rusia, mitra strategis Rusia, "
        "wilayah Timur Jauh Rusia, pelabuhan Vladivostok, industri Rusia, "
        "hubungan diplomatik dengan Rusia, kunjungan pejabat asing ke Rusia, "
        "Kremlin Moskow, Lapangan Merah, Istana Kremlin, juru bicara Kremlin, Dmitry Peskov, "
        "Sergey Lavrov, Menlu Rusia, diplomasi Rusia, Rusia dan negara Asia, Rusia dan ASEAN, "
        "Rusia dan Asia Tenggara, kunjungan kepala negara ke Rusia, tamu negara Rusia, "
        "ekonomi Rusia pasca sanksi, dampak sanksi terhadap Rusia, Rusia dan China, "
        "Rusia dan India, Rusia di panggung internasional, isolasi Rusia, Rusia G20"
    ),
    "lainnya": "topik di luar dua kategori di atas",
}

# ---------------------------------------------------------------
# Model & parameter LLM
# ---------------------------------------------------------------
from google import genai
from google.genai import types

GEMINI_MODEL = "gemini-3.1-flash-lite"
gemini_client = genai.Client(api_key=os.environ["GOOOGLE_API_KEY"])

SLEEP_BETWEEN_CALLS = 4

# Bobot skor ranking top author (harus berjumlah 1.0)
WEIGHT_POST_COUNT = 0.6
WEIGHT_ENGAGEMENT = 0.25
WEIGHT_FOLLOWERS = 0.15

# ---------------------------------------------------------------
# Parameter batch & ranking
# ---------------------------------------------------------------
BATCH_SIZE_CLASSIFY = 40
BATCH_SIZE_NER = 20
MAX_WORKERS_NER = 4
TOP_N_AUTHORS_PER_THEME = 10
MAX_POSTS_PER_AUTHOR_SUMMARY = 15

print(f"\n[INFO] Mode aktif: {'RESUME (lanjut dari df_clean tersimpan)' if RESUME_FROM_THEME else 'FULL (mulai dari awal)'}")
print("Konfigurasi siap.")

[INFO] File ID terdeteksi: 1RWje0fuPtl1jHVzibr480gWfK0ivJkbY
Mengunduh file .xlsx dari Google Drive...


Downloading...
From: https://drive.google.com/uc?id=1RWje0fuPtl1jHVzibr480gWfK0ivJkbY
To: /content/data_input.xlsx
100%|██████████| 11.6M/11.6M [00:00<00:00, 120MB/s]



[INFO] Sheet yang tersedia di file ini: ['Sheet1']

[OK] 66406 baris, 13 kolom berhasil dimuat
Kolom: ['No', 'Type', 'Headline', 'Mentions', 'Date', 'Link', 'Media', 'Sentiment', 'Author', 'Followers', 'Retweeted', 'Favourited', 'Location']


,No,Type,Headline,Mentions,Date,Link,Media,Sentiment,Author,Followers,Retweeted,Favourited,Location
0,1,mention,NaN,Prabowo: Karhutla Jangan Sampai ke Zona Inti I...,2026-09-07 17:40:17,https://twitter.com/web/statuses/2096911425180...,Twitter,Positive,@kompascom,8042391,0,0,Jakarta
1,2,mention,NaN,Presiden Prabowo Subianto memberikan instruksi...,2026-09-07 17:33:56,https://twitter.com/web/statuses/2096909830162...,Twitter,Positive,@makcrigis_,177,0,0,NaN
2,3,mention,Prabowo Tegas Larang Segala Bentuk Pembakaran ...,Jakarta: Presiden RI Prabowo Subianto menginst...,2026-09-07 17:33:21,https://www.metrotvnews.com/read/b2lC62aV-prab...,News,Neutral,www.metrotvnews.com,0,0,0,NaN
3,4,mention,Prabowo Tegas Larang Segala Bentuk Pembakaran ...,Jakarta: Presiden RI Prabowo Subianto menginst...,2026-09-07 17:33:21,https://www.metrotvnews.com/read/b2lC62aV-prab...,News,Positive,www.metrotvnews.com,0,0,0,NaN
4,5,mention,NaN,Karhutla Kepulauan Meranti Tembus 133 Hektare ...,2026-09-07 17:33:20,https://twitter.com/web/statuses/2096909679599...,Twitter,Positive,@bukamata18,156,0,0,Jakarta


Konfigurasi siap.


In [5]:
# ---------------------------------------------------------------
# Fungsi bantu untuk klasifikasi tema (keyword + LLM few-shot, multi-label)
# ---------------------------------------------------------------

def keyword_match_theme(text):
    """Prefilter cepat berdasarkan keyword di THEME_DESCRIPTIONS, TIDAK dikirim ke LLM.
    Return SATU tema (single) atau None kalau tidak ketemu -> lanjut ke LLM."""
    text_lower = str(text).lower()
    for theme in THEMES:
        if theme == "lainnya":
            continue
        desc = THEME_DESCRIPTIONS[theme]
        kw_list = [re.escape(k.strip()) for k in desc.split(",") if k.strip()]
        pattern = "|".join(kw_list)
        if re.search(pattern, text_lower):
            return theme
    return None  # tidak ketemu keyword apapun, biar LLM yang putuskan


THEME_LIST_STR = "\n".join(f"- {t}: {THEME_DESCRIPTIONS[t]}" for t in THEMES)

FEWSHOT_THEME_EXAMPLES = """
Contoh klasifikasi (belajar dari pola ini):

Teks: "Presiden Prabowo tiba di Vladivostok untuk menghadiri Eastern Economic Forum dan bertemu Putin"
Jawaban: {"themes": ["kunjungan_prabowo", "rusia_putin"], "primary_theme": "kunjungan_prabowo", "confidence": 0.95}

Teks: "Putin sampaikan pidato di forum ekonomi Rusia soal kerja sama energi dengan negara Asia"
Jawaban: {"themes": ["rusia_putin"], "primary_theme": "rusia_putin", "confidence": 0.9}

Teks: "Prabowo dan Putin teken sejumlah nota kesepahaman kerja sama investasi di sela EEF Vladivostok"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.95}

Teks: "Rusia hadapi sanksi baru dari negara Barat terkait konflik dengan Ukraina"
Jawaban: {"themes": ["rusia_putin"], "primary_theme": "rusia_putin", "confidence": 0.85}

Teks: "Harga cabai naik jelang lebaran, pedagang mengeluh"
Jawaban: {"themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.95}

Teks: "Momen Putin sambut langsung Prabowo di Vladivostok dengan upacara kenegaraan"
Jawaban: {"themes": ["kunjungan_prabowo", "rusia_putin"], "primary_theme": "kunjungan_prabowo", "confidence": 0.95}

Teks: "Prabowo dan Putin gelar pertemuan bilateral di sela-sela EEF, bahas kerja sama pertahanan dan energi"
Jawaban: {"themes": ["kunjungan_prabowo", "rusia_putin"], "primary_theme": "kunjungan_prabowo", "confidence": 0.95}

Teks: "Hasil pertemuan Prabowo-Putin di Vladivostok: RI-Rusia sepakat perkuat investasi dan perdagangan"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9}

Teks: "Menlu Sugiono dampingi Prabowo dalam kunjungan ke Vladivostok temui Presiden Putin"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9}

Teks: "Kritik warganet soal kunjungan Prabowo ke Rusia temui Putin di tengah isu netralitas Indonesia"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.85}

Teks: "Vladivostok jadi tuan rumah Eastern Economic Forum, dihadiri sejumlah kepala negara Asia"
Jawaban: {"themes": ["rusia_putin"], "primary_theme": "rusia_putin", "confidence": 0.8}

Teks: "Prabowo pulang dari Rusia usai lawatan temui Putin, bawa sejumlah kesepakatan dagang"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9}

Teks: "wowo ke vladivostok ketemu putin, katanya mau bahas kerja sama militer wajah menangis"
Jawaban: {"themes": ["kunjungan_prabowo", "rusia_putin"], "primary_theme": "kunjungan_prabowo", "confidence": 0.85}

Teks: "RT ngapain sih presiden kita malah jalan2 ke Rusia, di sini rakyat susah cari kerja"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.8}

Teks: "prabowo ke vladivostok gabut apa emang ada kepentingan sih, kok ga umumin dari kemarin"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.75}

Teks: "putin emang jago diplomasi, sampe prabowo aja mau jauh2 ke vladivostok"
Jawaban: {"themes": ["kunjungan_prabowo", "rusia_putin"], "primary_theme": "kunjungan_prabowo", "confidence": 0.85}

Teks: "seneng liat foto prabowo sama putin pas jamuan makan siang di vladivostok, gagah banget bapak kita tepuk tangan"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9}

Teks: "kunjungan Prabowo ke Rusia dianggap penting buat diversifikasi kerja sama luar negeri RI"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9}

Teks: "prabowo emang selalu bikin gempar tiap kunjungan luar negeri, kemarin china sekarang rusia"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.75}

Teks: "gibran hari ini resmikan jalan tol baru di jawa tengah"
Jawaban: {"themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.9}

Teks: "prabowo tinjau lokasi bencana banjir di jawa barat, salurkan bantuan logistik"
Jawaban: {"themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.9}

Teks: "Prabowo dan Putin sarapan bisnis bareng di Pulau Russky sebelum forum ekonomi dimulai"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9}

Teks: "Dubes Rusia Sergei Tolchenov pastikan bakal ada pertemuan bilateral lagi antara Prabowo dan Putin"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.85}

Teks: "Penasihat Kremlin Yury Ushakov umumkan Prabowo jadi tamu utama EEF 2026 di Vladivostok"
Jawaban: {"themes": ["kunjungan_prabowo", "rusia_putin"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9}

Teks: "Prabowo dan Putin bahas kerja sama militer, energi nuklir, dan ekspor gandum Rusia ke Indonesia"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9}

Teks: "Prabowo minta maaf ke Putin karena tidak bisa hadir langsung di KTT ASEAN-Rusia bulan Juni kemarin"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.85}

Teks: "Sebelum ke Vladivostok, Prabowo dan Putin sempat ketemu di Kremlin Moskow awal tahun ini"
Jawaban: {"themes": ["kunjungan_prabowo", "rusia_putin"], "primary_theme": "kunjungan_prabowo", "confidence": 0.8}

Teks: "trafik motor macet parah di sekitar Kremlin depok gara-gara ada demo warga"
Jawaban: {"themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.85}
"""


def build_theme_prompt(batch_texts):
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(batch_texts))
    return f"""Kamu adalah classifier tema untuk cuitan/berita berbahasa Indonesia.

Daftar tema yang tersedia:
{THEME_LIST_STR}

{FEWSHOT_THEME_EXAMPLES}

Sekarang klasifikasikan SETIAP teks di bawah ini. SATU teks BOLEH punya lebih dari
1 tema kalau memang relevan (lihat contoh ke-1 di atas), tapi tetap
tentukan "primary_theme" sebagai tema yang paling dominan/utama dibahas.
Jika tidak cocok ke tema manapun selain "lainnya", gunakan "lainnya" saja.

Teks:
{numbered}

Jawab HANYA dengan JSON array (tanpa penjelasan, tanpa markdown code block), formatnya:
[
  {{"index": 1, "themes": ["kunjungan_prabowo", "rusia_putin"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9}},
  {{"index": 2, "themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.6}}
]
Jumlah item HARUS sama persis dengan jumlah teks di atas ({len(batch_texts)} item)."""


def classify_theme_batch(batch_texts, retry=6):
    prompt = build_theme_prompt(batch_texts)
    for attempt in range(retry):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=2000,
                ),
            )
            parsed = json.loads(resp.text.strip())
            if len(parsed) != len(batch_texts):
                raise ValueError(f"Jumlah hasil ({len(parsed)}) != jumlah input ({len(batch_texts)})")
            return parsed
        except Exception as e:
            err_str = str(e)
            is_quota_exhausted = "RESOURCE_EXHAUSTED" in err_str or "429" in err_str
            is_overload = "503" in err_str or "UNAVAILABLE" in err_str

            if is_quota_exhausted:
                # kalau limitnya harian (GenerateRequestsPerDayPerProjectPerModel),
                # nunggu detik saja percuma -> harus nunggu jauh lebih lama atau stop
                wait = 60 * (attempt + 1)
            elif is_overload:
                wait = 20 * (attempt + 1)
            else:
                wait = 5 * (attempt + 1)

            print(f"[WARN] Klasifikasi tema batch gagal (percobaan {attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    print(f"[ERROR] Batch ini gagal total setelah {retry} percobaan, dicap 'lainnya' sementara")
    return [
        {"index": i + 1, "themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.0}
        for i in range(len(batch_texts))
    ]


print("Fungsi klasifikasi tema siap (keyword prefilter + LLM few-shot multi-label).")

Fungsi klasifikasi tema siap (keyword prefilter + LLM few-shot multi-label).


## 4. Step 1 — Load Data dari Google Sheets

Output: tabel mentah + daftar kolom, buat konfirmasi data ke-load dengan benar.


In [6]:
# ---------------------------------------------------------------
# Step 1b - Deteksi & buang akun media, dijalankan SEBELUM cleaning teks
# supaya proses cleaning tidak buang waktu di baris yang toh dibuang juga
# ---------------------------------------------------------------
import re

MEDIA_TYPE_ALWAYS = {"news"}
DOMAIN_SUFFIXES = (".com", ".id", ".co", ".net", ".org")
DOMAIN_TEXT_SUFFIXES = ("dotcom", "dotid", "dotco", "dotnet", "dotorg")  # banyak outlet nulis domain sbg teks

# ---------------------------------------------------------------
# Whitelist & keyword akun media
# SATU-SATUNYA sumber MEDIA_ACCOUNTS -> jangan didefinisikan ulang di blok lain,
# supaya tidak ada akun yang "hilang" gara-gara ketimpa definisi kedua.
# nama akun disimpan dalam bentuk sudah dinormalisasi: huruf kecil, tanpa "@", tanpa simbol/angka
# ---------------------------------------------------------------
MEDIA_ACCOUNTS = {
    # nasional
    "detikcom", "kompascom", "cnnindonesia", "tempodotco", "antaranews",
    "cnbcindonesia", "liputan6dotcom", "liputan6", "sctv", "tvonenews", "kumparan",
    "beritasatu", "metrotvnews", "tribunnews", "republikaonline", "suaradotcom",
    "jawapos", "bbcindonesia", "voaindonesia", "narasitv", "sindonews",
    "vivacoid", "bisniscom", "hariankompas", "tirtoid", "alineadotid",
    "inilahcom", "merdekadotcom", "okezone", "idntimes", "medcomid",
    "rri", "tvri", "jpnndotcom", "grid", "inewsdotid", "fajar",
    "pikiranrakyat", "gatra", "nuonline", "radioelshinta", "mediaindonesia",
    "setkabgoid",  # akun humas resmi Sekretariat Kabinet -> media pemerintah, bukan personal
    # malaysia
    "awani", "bharianmy", "bernamadotcom", "utusandotcom", "sinarharian",
    "malaysiakini", "thestar", "nst", "theedgemarkets",
    # media Rusia / terkait tema rusia_putin -> penting supaya tidak dianggap "organic author"
    "rt", "actualidadrt", "rtcom", "sputnik", "sputniknews", "sputnikindonesia",
    "tass", "tassagency", "rianovosti", "ria", "kremlin", "kremlinru",
    "telesur", "telesurtv",
}
MEDIA_KEYWORDS_SUBSTRING = [
    "news", "media", "redaksi", "newsroom", "official", "humas", "koran",
    "awani", "gazette", "tv", "radio", "pers",
    # NOTE: keyword institusi pemerintah (kemen, bnpb, bmkg, bpbd, polri, setneg)
    # SENGAJA tidak dimasukkan -> akun kementerian/lembaga tetap dihitung sbg
    # kandidat top author, tidak dianggap "media".
    # NOTE: "pers" sebagai substring bisa nyerempet nama seperti "persija", "persib" (klub bola)
    # -> kalau data kamu banyak bahas sepak bola, pindahkan "pers" balik ke EXACT list di bawah.
]
# kata kunci PENDEK/AMBIGU sisanya -> tetap exact-match per kata
MEDIA_KEYWORDS_EXACT = []


def normalize_account(text: str) -> str:
    """Huruf kecil + angka saja, buat cocokin ke whitelist/domain suffix.
    PENTING: jangan buang digit -> banyak nama outlet pakai angka, mis. 'liputan6dotcom'."""
    return re.sub(r"[^a-z0-9]", "", str(text).strip().lower())


def contains_media_keyword(author_lower: str) -> bool:
    # cek substring dulu (keyword yang aman/spesifik, termasuk tv/radio/pers sekarang)
    if any(kw in author_lower for kw in MEDIA_KEYWORDS_SUBSTRING):
        return True
    # exact-token untuk sisa keyword yang masih rawan ambigu (saat ini kosong)
    tokens = re.split(r"[^a-z]+", author_lower)
    return any(tok in MEDIA_KEYWORDS_EXACT for tok in tokens if tok)


def is_media_account(row) -> bool:
    media_val = str(row.get(COL_MEDIA, "")).strip().lower()
    author_raw = str(row.get(COL_AUTHOR_ID, "")).strip().lower().lstrip("@")
    author_norm = normalize_account(author_raw)

    if media_val in MEDIA_TYPE_ALWAYS:
        return True
    if author_norm in MEDIA_ACCOUNTS:
        return True
    if contains_media_keyword(author_raw):
        return True
    if author_raw.endswith(DOMAIN_SUFFIXES):
        return True
    if author_norm.endswith(DOMAIN_TEXT_SUFFIXES):
        return True
    return False


# pastikan Followers numeric dulu, biar sort_values di QC akurat
df_raw[COL_FOLLOWERS] = pd.to_numeric(df_raw.get(COL_FOLLOWERS, 0), errors="coerce").fillna(0)

# simpan salinan data mentah utuh dulu, jaga-jaga kalau nanti butuh cek ulang baris media
df_raw_original = df_raw.copy()

df_raw["is_media"] = df_raw.apply(is_media_account, axis=1)

n_media = df_raw["is_media"].sum()
n_nonmedia = (~df_raw["is_media"]).sum()
print(f"[INFO] {n_media} baris media (dibuang SEBELUM cleaning)")
print(f"[INFO] {n_nonmedia} baris non-media (lanjut ke cleaning)")

print("\nContoh deteksi per akun unik:")
display(
    df_raw[[COL_AUTHOR_ID, COL_MEDIA, "is_media"]]
    .drop_duplicates(subset=COL_AUTHOR_ID)
    .head(20)
)

print("\n[QC 1] Followers tertinggi yang KE-FLAG MEDIA — cek jangan sampai personal/influencer besar salah kena filter:")
display(
    df_raw[df_raw["is_media"]]
    .drop_duplicates(subset=COL_AUTHOR_ID)
    .sort_values(COL_FOLLOWERS, ascending=False)
    [[COL_AUTHOR_ID, COL_MEDIA, COL_FOLLOWERS]]
    .head(15)
)

print("\n[QC 2] Followers tertinggi yang TIDAK ke-flag — cek jangan sampai media lolos heuristik:")
display(
    df_raw[~df_raw["is_media"]]
    .drop_duplicates(subset=COL_AUTHOR_ID)
    .sort_values(COL_FOLLOWERS, ascending=False)
    [[COL_AUTHOR_ID, COL_MEDIA, COL_FOLLOWERS]]
    .head(15)
)

df_raw = df_raw[~df_raw["is_media"]].drop(columns=["is_media"]).reset_index(drop=True)
print(f"\n[INFO] df_raw sekarang berisi {len(df_raw)} baris non-media, siap dibersihkan")

[INFO] 20420 baris media (dibuang SEBELUM cleaning)
[INFO] 45986 baris non-media (lanjut ke cleaning)

Contoh deteksi per akun unik:


,Author,Media,is_media
0,@kompascom,Twitter,True
1,@makcrigis_,Twitter,False
2,www.metrotvnews.com,News,True
4,@bukamata18,Twitter,False
5,@KolektorBatu,Twitter,False
6,news.detik.com,News,True
7,www.liputan6.com,News,True
9,Tribun MedanTV,Youtube,True
10,riauterkini.com,News,True
14,20.detik.com,News,True



[QC 1] Followers tertinggi yang KE-FLAG MEDIA — cek jangan sampai personal/influencer besar salah kena filter:


,Author,Media,Followers
39761,@detikcom,Twitter,23651625
33931,officialinews,Tiktok,12400000
10942,metro_tv,Tiktok,9900000
7215,inilahcom,Tiktok,9800000
13752,@Metro_TV,Twitter,8753399
37598,@tvOneNews,Twitter,8083610
0,@kompascom,Twitter,8042391
20069,kompascom,Tiktok,7200000
30970,detikcom,Tiktok,6300000
22180,tribunnews,Tiktok,5400000



[QC 2] Followers tertinggi yang TIDAK ke-flag — cek jangan sampai media lolos heuristik:


,Author,Media,Followers
29151,@business,Twitter,10433736
15578,@grok,Twitter,9036980
41195,@MarioNawfal,Twitter,3886617
64957,@susipudjiastuti,Twitter,3555891
41844,@ruhutsitompul,Twitter,2304083
39465,merdekacom,Tiktok,2300000
26054,@fadlizon,Twitter,1757289
33717,@Fahrihamzah,Twitter,1524435
4980,@SinarOnline,Twitter,1103580
43347,@jakpost,Twitter,1061413



[INFO] df_raw sekarang berisi 45986 baris non-media, siap dibersihkan


## 5. Step 2 — Cleaning Teks

Menggabungkan `Headline` + `Mentions`, lalu membersihkan: URL, mention,
elongasi huruf ("parahhhh" -> "parah"), emoji -> teks, normalisasi kata alay,
dan dedup post identik/retweet.

### 5a. Fungsi cleaning


In [7]:
KAMUS_ALAY = {
    "gak": "tidak", "ga": "tidak", "nggak": "tidak", "tdk": "tidak", "gk": "tidak",
    "kaga": "tidak", "kagak": "tidak", "ngga": "tidak", "enggak": "tidak", "tak": "tidak",
    "gaada": "tidak ada", "bgt": "banget", "bnget": "banget", "bgttt": "banget",
    "bgt2": "banget", "banget2": "banget", "sangattt": "sangat",
    "yg": "yang", "yng": "yang", "krn": "karena", "krna": "karena", "karna": "karena",
    "sm": "sama", "sm2": "sama-sama", "sama2": "sama-sama",
    "utk": "untuk", "u/": "untuk", "bwt": "buat", "buatt": "buat", "utuk": "untuk",
    "dr": "dari", "drpd": "daripada",
    "skrg": "sekarang", "skrng": "sekarang", "skarang": "sekarang", "skg": "sekarang",
    "dgn": "dengan", "dg": "dengan", "dngan": "dengan",
    "org": "orang", "orng": "orang",
    "tp": "tapi", "tpi": "tapi", "spt": "seperti", "kayak": "seperti", "kaya": "seperti", "kyk": "seperti",
    "blm": "belum", "jgn": "jangan", "jgnkan": "jangankan", "jgnlah": "janganlah",
    "emg": "memang", "emang": "memang", "emank": "memang", "emgnya": "memangnya", "emangnya": "memangnya",
    "jd": "jadi", "jadinya": "jadinya", "jg": "juga", "aja": "saja", "aj": "saja",
    "udah": "sudah", "udh": "sudah", "dah": "sudah", "sdh": "sudah",
    "gmn": "bagaimana", "gmna": "bagaimana", "gmana": "bagaimana", "gimana": "bagaimana",
    "knp": "kenapa", "knpa": "kenapa", "napa": "kenapa", "ngapain": "sedang apa", "ngapa": "kenapa",
    "kalo": "kalau", "klo": "kalau",
    "trs": "terus", "trus": "terus", "gini": "begini", "gitu": "begitu", "gt": "begitu", "gtu": "begitu",
    "sy": "saya", "km": "kamu", "gw": "saya", "gwa": "saya", "gua": "saya", "gue": "saya",
    "ane": "saya", "aq": "saya", "aku": "saya",
    "lo": "kamu", "lu": "kamu", "elu": "kamu", "elo": "kamu", "ente": "kamu", "situ": "kamu",
    "hrs": "harus", "harus2": "harus", "kudu": "harus", "musti": "harus",
    "bs": "bisa", "bisa2": "bisa", "msh": "masih", "dlm": "dalam",
    "sblm": "sebelum", "stlh": "setelah", "pd": "pada", "pgn": "ingin", "pengen": "ingin",
    "liat": "lihat", "abis": "habis", "bkn": "bukan",
    "gpp": "tidak apa-apa", "gapapa": "tidak apa-apa",
    "cmn": "cuma", "cuman": "cuma", "mksh": "terima kasih", "makasih": "terima kasih",
    "moga": "semoga", "smoga": "semoga",
    "wkt": "waktu", "cpt": "cepat", "lg": "lagi", "lgi": "lagi",
    "sll": "selalu", "sllu": "selalu", "prnh": "pernah",
    "sndiri": "sendiri", "stiap": "setiap", "byk": "banyak", "dikit": "sedikit",
    "dtg": "datang", "krg": "kurang", "ato": "atau",
    "pke": "pakai", "pake": "pakai", "denger": "dengar", "kasih": "beri", "bikin": "buat",
    "brp": "berapa", "walopun": "walaupun", "walaupun": "walaupun", "meskipun": "meskipun", "meski": "meskipun",
    "kayanya": "sepertinya", "kykny": "sepertinya",
    "dpt": "dapat", "dapet": "dapat", "tggl": "tinggal", "tinggl": "tinggal",
    "tmn": "teman", "temen": "teman",
    "trnyata": "ternyata", "ternyta": "ternyata",
    "sbnrnya": "sebenarnya", "sebenernya": "sebenarnya", "sbnernya": "sebenarnya",
    "sbg": "sebagai", "sbgai": "sebagai", "trhdp": "terhadap", "thd": "terhadap", "thdp": "terhadap",
    "diantaranya": "di antaranya", "diantara": "di antara",
    "ngerti": "mengerti", "ngerasa": "merasa", "berasa": "terasa",
    "keliatan": "terlihat", "keliatannya": "terlihatnya",
    "nyari": "mencari", "nyoba": "mencoba", "nunggu": "menunggu",
    "ngasih": "memberi", "ngajak": "mengajak", "ngobrol": "berbicara",
    "nyadar": "sadar", "ngerasain": "merasakan", "ngebayangin": "membayangkan",
    "mikir": "berpikir", "mikirin": "memikirkan", "ngomongin": "membicarakan",
    "kesel": "kesal", "sebel": "sebal", "capek": "lelah", "cape": "lelah",
    "males": "malas", "mager": "malas gerak", "seneng": "senang",
    "parah": "sangat", "gila": "sangat", "anjir": "sangat", "anjay": "sangat",
    "mantap": "bagus", "mantul": "bagus", "keren": "bagus",
    "jelek": "buruk", "ancur": "hancur", "hancur": "hancur",
    "ngeri": "mengerikan", "serem": "menyeramkan",
    "bego": "bodoh", "goblok": "bodoh", "tolol": "bodoh", "bodo": "bodoh",
    "songong": "sombong", "belagu": "sombong",
    "curhat": "curahan hati", "japri": "pesan pribadi",
    "bener": "benar", "beneran": "benaran", "makanya": "makanya", "makannya": "makanya",
    "makin": "semakin", "kian": "semakin",
    "besok": "besok", "bsk": "besok", "kmrn": "kemarin", "kmarin": "kemarin", "kemaren": "kemarin",
    "td": "tadi", "entar": "nanti", "ntar": "nanti", "nti": "nanti",
    "pemrintah": "pemerintah", "pemerintahan": "pemerintah",
    "korup": "korupsi", "dikorupsi": "korupsi", "ngorupsi": "korupsi",
    "nyolong": "mencuri", "maling": "pencuri",
    "boong": "bohong", "bohong2": "bohong", "hoax": "hoaks", "hoak": "hoaks",
    "settingan": "rekayasa", "settingannya": "rekayasa", "php": "janji palsu",
    "ngamuk": "marah", "murka": "marah", "demo": "demonstrasi",
    "smpai": "sampai", "sampe": "sampai",
    "tuhh": "tuh", "sihh": "sih", "dehh": "deh",
    "pak": "bapak", "bu": "ibu", "min": "admin",
}

RE_URL = re.compile(r"(https?://\S+|www\.\S+)")
RE_MENTION = re.compile(r"@[A-Za-z0-9_]+")
RE_HASHTAG = re.compile(r"#([A-Za-z0-9_]+)")
RE_RT_PREFIX = re.compile(r"^\s*RT\s*@[A-Za-z0-9_]+\s*:\s*", flags=re.IGNORECASE)
RE_ELONGATION = re.compile(r"(.)\1{2,}")
RE_MULTI_SPACE = re.compile(r"\s+")
RE_NON_ALNUM_PUNCT = re.compile(r"[^\w\s.,!?]")


def demojize_or_strip(text: str) -> str:
    if HAS_EMOJI_LIB:
        text = emoji.demojize(text, language="id" if "id" in emoji.LANGUAGES else "en")
        return text.replace("_", " ").replace(":", " ")
    return text


def fix_elongation(text: str) -> str:
    return RE_ELONGATION.sub(r"\1\1", text)


def normalize_slang(text: str) -> str:
    words = text.split()
    return " ".join(KAMUS_ALAY.get(w.lower().strip(".,!?"), w) for w in words)


def extract_hashtags(text: str):
    return RE_HASHTAG.findall(text)


def extract_mentions(text: str):
    return RE_MENTION.findall(text)


def build_raw_text(headline, mentions) -> str:
    """Gabungkan Headline + Mentions jadi satu teks mentah."""
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if not headline or headline.lower() == mentions.lower() or headline.lower() == "nan":
        return mentions or headline
    if not mentions:
        return headline
    return f"{headline}. {mentions}"


def clean_text(raw: str) -> str:
    if not isinstance(raw, str) or not raw.strip():
        return ""
    text = raw
    text = RE_RT_PREFIX.sub("", text)
    text = RE_URL.sub(" ", text)
    text = RE_MENTION.sub(" ", text)
    text = RE_HASHTAG.sub(r"\1", text)
    text = demojize_or_strip(text)
    text = RE_NON_ALNUM_PUNCT.sub(" ", text)
    text = fix_elongation(text)
    text = normalize_slang(text)
    text = RE_MULTI_SPACE.sub(" ", text).strip()
    return text


def make_dedup_key(text_clean: str) -> str:
    key = re.sub(r"[^\w\s]", "", text_clean.lower())
    key = RE_MULTI_SPACE.sub(" ", key).strip()
    return hashlib.md5(key.encode("utf-8")).hexdigest()

print("Fungsi cleaning siap.")


Fungsi cleaning siap.


### 5b. Jalankan cleaning

In [8]:
headline_col = df_raw[COL_HEADLINE] if COL_HEADLINE in df_raw.columns else pd.Series([""] * len(df_raw))
mentions_col = df_raw[COL_MENTIONS] if COL_MENTIONS in df_raw.columns else pd.Series([""] * len(df_raw))
df_raw["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(headline_col, mentions_col)]

tqdm.pandas(desc="Cleaning teks")
df_raw["hashtags"] = df_raw["text_raw_combined"].astype(str).apply(extract_hashtags)
df_raw["mentions_akun"] = df_raw["text_raw_combined"].astype(str).apply(extract_mentions)
df_raw["text_clean"] = df_raw["text_raw_combined"].astype(str).progress_apply(clean_text)

before = len(df_raw)
df_clean = df_raw[df_raw["text_clean"].str.split().str.len().fillna(0) >= 3].copy()
print(f"[INFO] Buang {before - len(df_clean)} baris teks kosong/terlalu pendek setelah cleaning")

df_clean["dedup_key"] = df_clean["text_clean"].apply(make_dedup_key)
before = len(df_clean)
df_clean["is_duplicate"] = df_clean.duplicated(subset="dedup_key", keep="first")
n_dup = df_clean["is_duplicate"].sum()
df_clean = df_clean[~df_clean["is_duplicate"]].drop(columns=["is_duplicate", "dedup_key"])
df_clean = df_clean.reset_index(drop=True)  # index rapi 0..n, dipakai di step-step berikutnya

print(f"[INFO] Buang {n_dup} duplikat/retweet identik")
print(f"[OK] {len(df_clean)} baris tersisa setelah cleaning\n")

print("Contoh sebelum vs sesudah cleaning:")
display(df_clean[["text_raw_combined", "text_clean"]].head(5))

Cleaning teks:   0%|          | 0/45986 [00:00<?, ?it/s]

[INFO] Buang 1542 baris teks kosong/terlalu pendek setelah cleaning
[INFO] Buang 27067 duplikat/retweet identik
[OK] 17377 baris tersisa setelah cleaning

Contoh sebelum vs sesudah cleaning:


,text_raw_combined,text_clean
0,Presiden Prabowo Subianto memberikan instruksi...,Presiden Prabowo Subianto memberikan instruksi...
1,Karhutla Kepulauan Meranti Tembus 133 Hektare ...,Karhutla Kepulauan Meranti Tembus 133 Hektare ...
2,Presiden Prabowo mengatakan kapal induk dr Ita...,Presiden Prabowo mengatakan kapal induk dari I...
3,"Erupsi anak Krakatau, Presiden Prabowo perinta...","Erupsi anak Krakatau, Presiden Prabowo perinta..."
4,RT Kapolri Listyo Sigit Prabowo sambut kedatan...,RT Kapolri Listyo Sigit Prabowo sambut kedatan...


## 6. Step 3 — Klasifikasi Tema (LLM)

Dikirim per-batch (default 15 post/panggilan) supaya hemat API call.

### 6a. Fungsi klasifikasi


In [9]:
# ---------------------------------------------------------------
# Keyword list KHUSUS untuk prefilter cepat (terpisah dari THEME_DESCRIPTIONS
# yang dipakai buat prompt LLM). Sengaja dibikin pendek & high-recall.
# ---------------------------------------------------------------

# Entitas yang menandakan "Prabowo" (siapapun yang mewakili beliau/kunjungan)
KW_PRABOWO = [
    "prabowo", "presiden ri", "presiden indonesia", "presiden prabowo",
    "psubianto", "pak prabowo", "prabowosubianto", "#prabowo",
    "presiden subianto", "kepala negara ri", "kepala negara indonesia",
    "wowo", "pak wowo",
]

# Entitas yang menandakan konteks "Rusia" (tempat/tokoh/acara)
KW_RUSIA_KONTEKS = [
    "rusia", "russia", "putin", "vladimir putin", "kremlin", "moskow", "moscow",
    "vladivostok", "eastern economic forum", "eef2025", "eef 2025", " eef ",
    "#eef", "rosatom", "lavrov", "peskov", "#rusia", "#putin", "#vladivostok",
]

# Kata kunci umum topik Rusia tanpa embel2 kunjungan (buat tema rusia_putin)
KW_RUSIA_PUTIN_UMUM = KW_RUSIA_KONTEKS + [
    "ukraina", "brics", "sanksi rusia", "sanksi barat", "#brics",
]

# Kata kunci topik DOMESTIK yang menandakan mention "Prabowo" TIDAK ada
# hubungannya sama sekali dengan Rusia -> short-circuit ke "lainnya" tanpa LLM.
# Berdasarkan sampling manual dari data (isu makan bergizi gratis, elektabilitas,
# Gibran, agama, TNI dalam negeri, dsb).
KW_DOMESTIK_EXCLUDE = [
    "gibran", "apbn", "elektabilitas", "makan bergizi", "mbg",
    "pilkada", "pemilu", "kades", "psi", "gerindra", "kpk", "kdm",
    "muktamar", "ormas", "grib", "habib rizieq", "islam ala prabowo",
    "desil", "kemensos", "baznas", "mui", "salat jumat", "ramadhan",
    "papua", "kalimantan", "jawa", "listrik", "pln",
]


def _contains_any(text_lower, kw_list):
    return any(kw in text_lower for kw in kw_list)


def keyword_match_theme(text):
    """Prefilter cepat berbasis co-occurrence entitas, TIDAK dikirim ke LLM.
    Return SATU tema (single) atau None kalau tidak ketemu -> lanjut ke LLM."""
    text_lower = str(text).lower()

    ada_prabowo = _contains_any(text_lower, KW_PRABOWO)
    ada_rusia_konteks = _contains_any(text_lower, KW_RUSIA_KONTEKS)

    # kunjungan_prabowo: butuh Prabowo + konteks Rusia muncul BARENGAN di teks yang sama
    if ada_prabowo and ada_rusia_konteks:
        return "kunjungan_prabowo"

    # rusia_putin: konteks Rusia muncul TANPA Prabowo
    if not ada_prabowo and _contains_any(text_lower, KW_RUSIA_PUTIN_UMUM):
        return "rusia_putin"

    # short-circuit: ada Prabowo tapi jelas topik domestik & TIDAK ada singgungan
    # Rusia sama sekali -> aman dicap "lainnya" tanpa perlu LLM
    if ada_prabowo and not ada_rusia_konteks and _contains_any(text_lower, KW_DOMESTIK_EXCLUDE):
        return "lainnya"

    return None  # masih ambigu -> biar LLM yang putuskan


print("Keyword prefilter (versi fix + exclude list) siap.")

Keyword prefilter (versi fix + exclude list) siap.


In [ ]:
# ---------------------------------------------------------------
# Step 3 - Klasifikasi Tema (hybrid: keyword dulu, sisanya baru LLM few-shot + multi-label)
# ---------------------------------------------------------------

TEST_MODE = False
N_TEST_ROWS = 100

# simpan salinan data bersih yang tidak akan pernah ketimpa lagi
# (df_clean di sini sudah pasti non-media, karena media sudah dibuang di Step 1b)
if "df_master_clean" not in globals():
    df_master_clean = df_clean.copy()
    print(f"[INFO] df_master_clean dibuat, {len(df_master_clean)} baris")
else:
    print(f"[INFO] df_master_clean sudah ada, {len(df_master_clean)} baris (tidak dibuat ulang)")

if TEST_MODE:
    df_run = df_master_clean.sample(n=min(N_TEST_ROWS, len(df_master_clean)), random_state=42).reset_index(drop=True)
    print(f"[TEST MODE] Memakai {len(df_run)} baris dari total {len(df_master_clean)} baris")
else:
    df_run = df_master_clean.copy()
    print(f"[FULL MODE] Memakai SEMUA {len(df_run)} baris")

texts_run = df_run["text_clean"].fillna("").tolist()
themes_run = [None] * len(texts_run)          # primary_theme (single, dipakai Step 5 ranking)
all_themes_run = [None] * len(texts_run)       # semua tema relevan (list, insight tambahan)
confidences_run = [0.0] * len(texts_run)

# tahap 1: keyword matching
for i, t in enumerate(texts_run):
    match = keyword_match_theme(t)
    if match:
        themes_run[i] = match
        all_themes_run[i] = [match]
        confidences_run[i] = 1.0  # keyword cocok persis, confidence dianggap tinggi

n_keyword = sum(1 for th in themes_run if th is not None)
need_llm_idx = [i for i, th in enumerate(themes_run) if th is None]
print(f"[INFO] {n_keyword} baris langsung kena keyword")
print(f"[INFO] {len(need_llm_idx)} baris dikirim ke LLM karena tidak ketemu keyword")

# tahap 2: sisanya baru dikirim ke LLM (few-shot + multi-label)
texts_for_llm = [texts_run[i] for i in need_llm_idx]

n_batches = (len(texts_for_llm) + BATCH_SIZE_CLASSIFY - 1) // BATCH_SIZE_CLASSIFY
for b in tqdm(range(n_batches), desc="Klasifikasi tema (LLM)"):
    start = b * BATCH_SIZE_CLASSIFY
    end = start + BATCH_SIZE_CLASSIFY
    batch = texts_for_llm[start:end]
    results = classify_theme_batch(batch)
    for r in results:
        local_idx = r["index"] - 1  # posisi DI DALAM batch ini saja (0..len(batch)-1)
        if 0 <= local_idx < len(batch):
            global_idx = need_llm_idx[start + local_idx]
            themes_raw = r.get("themes", ["lainnya"])
            themes_valid = [t for t in themes_raw if t in THEMES] or ["lainnya"]
            primary = r.get("primary_theme", themes_valid[0])
            themes_run[global_idx] = primary if primary in THEMES else themes_valid[0]
            all_themes_run[global_idx] = themes_valid
            confidences_run[global_idx] = r.get("confidence", 0.5)
    time.sleep(SLEEP_BETWEEN_CALLS)

# jaga-jaga kalau masih ada None (harusnya tidak terjadi)
themes_run = [th if th is not None else "lainnya" for th in themes_run]
all_themes_run = [at if at is not None else ["lainnya"] for at in all_themes_run]

df_run["theme"] = themes_run
df_run["all_themes"] = all_themes_run
df_run["theme_confidence"] = confidences_run
df_run["theme"] = df_run["theme"].fillna("lainnya")
df_run["theme_confidence"] = df_run["theme_confidence"].fillna(0.0)

print("\nDistribusi tema (primary):")
display(df_run["theme"].value_counts())

print("\nContoh teks dengan multi-tema (dari hasil LLM):")
display(df_run[df_run["all_themes"].apply(len) > 1][["text_clean", "all_themes", "theme"]].head(10))

display(df_run[["text_clean", "theme", "all_themes", "theme_confidence"]].head(10))

df_clean = df_run.copy()
print(f"\n[INFO] df_clean sekarang berisi {len(df_clean)} baris")

[INFO] df_master_clean dibuat, 17377 baris
[FULL MODE] Memakai SEMUA 17377 baris
[INFO] 5325 baris langsung kena keyword
[INFO] 12052 baris dikirim ke LLM karena tidak ketemu keyword


Klasifikasi tema (LLM):   0%|          | 0/302 [00:00<?, ?it/s]

[WARN] Klasifikasi tema batch gagal (percobaan 1/6): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}} -> tunggu 20s
[WARN] Klasifikasi tema batch gagal (percobaan 1/6): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}} -> tunggu 20s
[WARN] Klasifikasi tema batch gagal (percobaan 2/6): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}} -> tunggu 40s
[WARN] Klasifikasi tema batch gagal (percobaan 1/6): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again l

In [ ]:
SAVE_PATH_AFTER_THEME = "/content/drive/MyDrive/df_clean_after_theme.pkl"

from google.colab import drive
drive.mount('/content/drive')

df_clean.to_pickle(SAVE_PATH_AFTER_THEME)
print(f"[OK] df_clean tersimpan ke {SAVE_PATH_AFTER_THEME}, {len(df_clean)} baris")

### 6b. Jalankan klasifikasi tema

In [ ]:
# ---------------------------------------------------------------
# TEST: ambil sampel dari SEMUA tema sekaligus (kunjungan_prabowo, rusia_putin, + lainnya)
# Pakai keyword_match_theme() yang sama dengan yang dipakai di Step 3,
# supaya sample ini benar-benar representasikan hasil prefilter asli.
# ---------------------------------------------------------------

N_PER_THEME = 5

# jalankan keyword_match_theme ke seluruh df_clean, kelompokkan berdasarkan hasilnya
theme_matches = {"kunjungan_prabowo": [], "rusia_putin": [], "lainnya": []}
for idx, text in df_clean["text_clean"].items():
    match = keyword_match_theme(text)
    if match in theme_matches:
        theme_matches[match].append(idx)
    else:
        theme_matches["lainnya"].append(idx)  # None (ambigu) dikumpulkan sbg kontrol juga

sample_parts = []
for theme in ["kunjungan_prabowo", "rusia_putin"]:
    idxs = theme_matches[theme]
    if idxs:
        n = min(N_PER_THEME, len(idxs))
        sample_parts.append(df_clean.loc[idxs].sample(n, random_state=42))
        print(f"[INFO] Tema '{theme}': ditemukan {len(idxs)} kandidat via keyword, ambil {n}")
    else:
        print(f"[WARN] Tema '{theme}': tidak ada kandidat yang cocok keyword")

# kontrol "lainnya" / ambigu (tidak kena keyword sama sekali)
idxs_lainnya = theme_matches["lainnya"]
n_lainnya = min(N_PER_THEME, len(idxs_lainnya))
sample_parts.append(df_clean.loc[idxs_lainnya].sample(n_lainnya, random_state=99))
print(f"[INFO] Kontrol 'lainnya'/ambigu: {len(idxs_lainnya)} kandidat, ambil {n_lainnya}")

df_test = pd.concat(sample_parts).drop_duplicates(subset="text_clean").reset_index(drop=True)
test_texts = df_test["text_clean"].tolist()

print(f"\n[TEST] Total {len(df_test)} teks akan dites (multi-tema)")
display(df_test[["text_clean"]])

# kalau jumlahnya lebih dari BATCH_SIZE_CLASSIFY, tetap dipecah per batch spy tidak kepotong
n_batches = (len(test_texts) + BATCH_SIZE_CLASSIFY - 1) // BATCH_SIZE_CLASSIFY
all_results = []
for b in range(n_batches):
    start = b * BATCH_SIZE_CLASSIFY
    end = start + BATCH_SIZE_CLASSIFY
    batch = test_texts[start:end]
    results = classify_theme_batch(batch)
    for r in results:
        r["index"] += start  # geser index biar cocok sama posisi global di test_texts
    all_results.extend(results)
    time.sleep(SLEEP_BETWEEN_CALLS)

print("\nHasil klasifikasi per tema:")
for r in sorted(all_results, key=lambda x: x["index"]):
    idx = r["index"] - 1
    primary = r.get("primary_theme", "?")
    all_th = r.get("themes", [])
    conf = r.get("confidence", "?")
    print(f"[{primary:15s}] all_themes={all_th} (conf={conf}) -> {test_texts[idx][:90]}")

## 7. Step 4 — Filter Akun Media & NER




### 7b. NER — ekstraksi entitas (lokasi, instansi, tokoh)

Hanya dijalankan untuk baris **non-media** & tema **relevan** (bukan "lainnya")
supaya hemat biaya API. Hapus filter ini di cell bawah kalau butuh NER untuk semua baris.


In [ ]:
FEWSHOT_NER_EXAMPLES = """
Contoh ekstraksi (belajar dari pola ini):

Teks: "Presiden Prabowo bertemu Putin di Vladivostok dalam rangkaian Eastern Economic Forum"
Jawaban: {"lokasi": ["Vladivostok"], "instansi": ["Eastern Economic Forum"], "tokoh": ["Prabowo", "Putin"]}

Teks: "Prabowo hadiri KTT EEF 2026 di Pulau Russky, Vladivostok, sebagai tamu utama"
Jawaban: {"lokasi": ["Pulau Russky", "Vladivostok"], "instansi": ["EEF"], "tokoh": ["Prabowo"]}

Teks: "Kualitas udara memburuk"
Jawaban: {"lokasi": [], "instansi": [], "tokoh": []}

Teks: "Prabowo dan Putin sarapan bisnis bareng di Pulau Russky sebelum forum ekonomi dimulai"
Jawaban: {"lokasi": ["Pulau Russky"], "instansi": [], "tokoh": ["Prabowo", "Putin"]}

Teks: "Dubes Rusia untuk Indonesia, Sergei Tolchenov, konfirmasi jadwal pertemuan bilateral Prabowo-Putin"
Jawaban: {"lokasi": [], "instansi": [], "tokoh": ["Sergei Tolchenov", "Prabowo", "Putin"]}

Teks: "Penasihat kebijakan luar negeri Kremlin, Yury Ushakov, umumkan kehadiran Prabowo di EEF Vladivostok"
Jawaban: {"lokasi": ["Kremlin", "Vladivostok"], "instansi": ["EEF"], "tokoh": ["Yury Ushakov", "Prabowo"]}

Teks: "Selain Prabowo, PM Mongolia Uchral dan Wapres Myanmar juga hadiri EEF tahun ini"
Jawaban: {"lokasi": [], "instansi": ["EEF"], "tokoh": ["Prabowo", "Uchral"]}

Teks: "Prabowo dan Putin bahas kerja sama militer, energi nuklir, dan ekspor gandum Rusia ke Indonesia"
Jawaban: {"lokasi": ["Indonesia"], "instansi": [], "tokoh": ["Prabowo", "Putin"]}

Teks: "Sebelumnya Prabowo dan Putin sempat bertemu di Kremlin, Moskow, awal tahun ini"
Jawaban: {"lokasi": ["Kremlin", "Moskow"], "instansi": [], "tokoh": ["Prabowo", "Putin"]}

Teks: "wowo ke vladivostok ketemu putin lagi, katanya bahas kerja sama militer wajah menangis"
Jawaban: {"lokasi": ["Vladivostok"], "instansi": [], "tokoh": ["Prabowo", "Putin"]}

Teks: "seneng liat foto prabowo sama putin pas jamuan makan siang di vladivostok, gagah bapak kita tepuk tangan"
Jawaban: {"lokasi": ["Vladivostok"], "instansi": [], "tokoh": ["Prabowo", "Putin"]}

Teks: "prabowo minta maaf ke putin karena ga bisa hadir langsung pas KTT ASEAN-Rusia bulan Juni kemarin"
Jawaban: {"lokasi": [], "instansi": ["ASEAN"], "tokoh": ["Prabowo", "Putin"]}

Teks: "RT ngapain sih presiden kita malah jalan2 ke Vladivostok, di sini rakyat susah cari kerja"
Jawaban: {"lokasi": ["Vladivostok"], "instansi": [], "tokoh": ["Prabowo"]}

Teks: "gibran hari ini resmikan jalan tol baru di jawa tengah"
Jawaban: {"lokasi": ["Jawa Tengah"], "instansi": [], "tokoh": ["Gibran"]}

Teks: "Harga cabai naik jelang lebaran, pedagang mengeluh"
Jawaban: {"lokasi": [], "instansi": [], "tokoh": []}
"""


def build_ner_prompt(batch_texts):
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(batch_texts))
    return f"""Ekstrak entitas penting dari tiap teks berbahasa Indonesia berikut.
Kategori entitas:
- lokasi: nama negara/kota/tempat (misal "Rusia", "Vladivostok", "Kremlin")
- instansi: nama lembaga/organisasi/acara resmi (misal "Kementerian Luar Negeri", "Eastern Economic Forum", "BRICS")
- tokoh: nama orang yang disebut (pejabat, tokoh publik, dll)

{FEWSHOT_NER_EXAMPLES}

Sekarang ekstrak entitas dari teks berikut. Kalau tidak ada entitas di kategori
tertentu, gunakan array kosong (lihat contoh ke-3 di atas) — JANGAN mengarang entitas.

Teks:
{numbered}

Jawab HANYA dengan JSON array, tanpa penjelasan, tanpa markdown code block:
[
  {{"index": 1, "lokasi": ["..."], "instansi": ["..."], "tokoh": ["..."]}},
  {{"index": 2, "lokasi": [], "instansi": [], "tokoh": []}}
]
Jumlah item HARUS sama persis dengan jumlah teks ({len(batch_texts)} item)."""


def extract_ner_batch(batch_texts, retry=6):
    prompt = build_ner_prompt(batch_texts)
    for attempt in range(retry):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=2000,
                ),
            )
            parsed = json.loads(resp.text.strip())
            if len(parsed) != len(batch_texts):
                raise ValueError(f"Jumlah hasil ({len(parsed)}) != jumlah input ({len(batch_texts)})")
            return parsed, True
        except Exception as e:
            is_overload = "503" in str(e) or "UNAVAILABLE" in str(e)
            wait = 20 * (attempt + 1) if is_overload else 5 * (attempt + 1)
            print(f"[WARN] NER batch gagal (percobaan {attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    print(f"[ERROR] Batch NER ini gagal total setelah {retry} percobaan")
    return [{"index": i + 1, "lokasi": [], "instansi": [], "tokoh": []} for i in range(len(batch_texts))], False


target_idx = df_clean[df_clean["theme"] != "lainnya"].index.tolist()
print(f"Menjalankan NER untuk {len(target_idx)} baris (tema relevan) ...")

texts_all = df_clean["text_clean"].fillna("").tolist()
lokasi_col = [[] for _ in range(len(df_clean))]
instansi_col = [[] for _ in range(len(df_clean))]
tokoh_col = [[] for _ in range(len(df_clean))]
ner_failed_col = [False for _ in range(len(df_clean))]

n_batches = (len(target_idx) + BATCH_SIZE_NER - 1) // BATCH_SIZE_NER
for b in tqdm(range(n_batches), desc="NER ekstraksi entitas"):
    idx_batch = target_idx[b * BATCH_SIZE_NER:(b + 1) * BATCH_SIZE_NER]
    text_batch = [texts_all[i] for i in idx_batch]
    results, success = extract_ner_batch(text_batch)
    for r, df_idx in zip(results, idx_batch):
        lokasi_col[df_idx] = r.get("lokasi", [])
        instansi_col[df_idx] = r.get("instansi", [])
        tokoh_col[df_idx] = r.get("tokoh", [])
        ner_failed_col[df_idx] = not success
    time.sleep(SLEEP_BETWEEN_CALLS)

df_clean["entities_lokasi"] = lokasi_col
df_clean["entities_instansi"] = instansi_col
df_clean["entities_tokoh"] = tokoh_col
df_clean["ner_failed"] = ner_failed_col

n_failed = df_clean.loc[target_idx, "ner_failed"].sum()
if n_failed > 0:
    print(f"[WARN] {n_failed} baris gagal diproses NER setelah semua percobaan, entitasnya kosong bukan berarti tidak ada")

print("\nContoh hasil NER:")
display(df_clean.loc[target_idx, ["text_clean", "entities_lokasi", "entities_instansi", "entities_tokoh", "ner_failed"]].head(10))

## 8. Step 5 — Ranking Top Author per Tema

Hanya akun **non-media**. Skor = kombinasi jumlah post + engagement + followers
(bobot diatur di bagian Konfigurasi).


In [ ]:
df_topic = df_clean[df_clean["theme"] != "lainnya"].copy()
print(f"[INFO] {len(df_topic)} baris tersisa setelah filter tema relevan (media sudah dibuang sejak Step 1b)")

for col in [COL_FAVOURITED, COL_RETWEETED]:
    if col in df_topic.columns:
        df_topic[col] = pd.to_numeric(df_topic[col], errors="coerce").fillna(0)
    else:
        df_topic[col] = 0

if COL_FOLLOWERS in df_topic.columns:
    df_topic[COL_FOLLOWERS] = pd.to_numeric(df_topic[COL_FOLLOWERS], errors="coerce").fillna(0)
else:
    df_topic[COL_FOLLOWERS] = 0

df_topic["engagement"] = df_topic[COL_FAVOURITED] + df_topic[COL_RETWEETED]

top_authors_per_theme = {}

for theme in THEMES:
    if theme == "lainnya":
        continue
    sub = df_topic[df_topic["theme"] == theme]
    if sub.empty:
        print(f"[INFO] Tidak ada data untuk tema '{theme}'")
        continue

    agg = (
        sub.groupby(COL_AUTHOR_ID)
        .agg(
            jumlah_post=(COL_AUTHOR_ID, "count"),
            total_engagement=("engagement", "sum"),
            followers=(COL_FOLLOWERS, "max"),
        )
        .reset_index()
        .rename(columns={COL_AUTHOR_ID: "author_id"})
    )

    max_post = agg["jumlah_post"].max() or 1
    max_eng = agg["total_engagement"].max() or 1
    max_followers = agg["followers"].max() or 1
    agg["score"] = (
        WEIGHT_POST_COUNT * (agg["jumlah_post"] / max_post)
        + WEIGHT_ENGAGEMENT * (agg["total_engagement"] / max_eng)
        + WEIGHT_FOLLOWERS * (agg["followers"] / max_followers)
    )

    agg = agg.sort_values("score", ascending=False).head(TOP_N_AUTHORS_PER_THEME)
    agg.insert(0, "theme", theme)
    agg["rank"] = range(1, len(agg) + 1)
    top_authors_per_theme[theme] = agg

    print(f"\n=== Top Author: {theme} ===")
    display(agg[["rank", "author_id", "jumlah_post", "total_engagement", "followers", "score"]])

df_top_authors = (
    pd.concat(top_authors_per_theme.values(), ignore_index=True)
    if top_authors_per_theme else pd.DataFrame(columns=["theme", "author_id", "jumlah_post", "total_engagement", "followers", "score", "rank"])
)

## 9. Step 6 — Ringkasan & Sentimen per Top Author (LLM)

Untuk tiap top author, semua post-nya (tentang tema itu) dikirim ke LLM
untuk diringkas + ditentukan sentimen keseluruhannya. Sentimen per-post yang
sudah ada dari tool monitoring (`Sentiment`) ikut dikirim sebagai konteks
tambahan, bukan patokan mutlak.

### 9a. Fungsi summarization


In [ ]:
SENTIMENT_OPTIONS = ["positif", "negatif", "kontroversi"]


def build_summary_prompt(author_label, theme, texts, tool_sentiment_note=""):
    joined = "\n".join(f"- {t}" for t in texts)
    context_note = ""
    if tool_sentiment_note:
        context_note = (
            f"\nSebagai referensi tambahan (bukan patokan mutlak), tool monitoring sosial media "
            f"sebelumnya sudah memberi label sentimen per-post untuk akun ini dengan distribusi: "
            f"{tool_sentiment_note}. Gunakan ini hanya sebagai bahan pertimbangan, "
            f"keputusan akhir tetap berdasarkan isi teks yang kamu baca sendiri.\n"
        )
    return f"""Berikut kumpulan cuitan dari akun "{author_label}" tentang topik "{theme}":

{joined}
{context_note}
Tugas kamu:
1. Ringkas dalam 2-3 kalimat apa pandangan/narasi utama yang disampaikan akun ini soal topik tersebut.
2. Tentukan sentimen KESELURUHAN akun ini terhadap topik, pilih SATU dari: {", ".join(SENTIMENT_OPTIONS)}.
   - "positif" dipakai jika akun mendukung/memuji topik ini.
   - "negatif" dipakai jika akun mengkritik/menentang topik ini.
   - "kontroversi" dipakai jika pendapat akun ini memicu perdebatan/pro-kontra, menyampaikan klaim yang kontroversial, atau mencampur pujian dan kritik sekaligus.
3. Berikan alasan singkat (1 kalimat) untuk sentimen tersebut.

Jawab HANYA dengan JSON, tanpa penjelasan tambahan, tanpa markdown code block:
{{"summary": "...", "sentiment": "...", "reason": "..."}}"""


def summarize_author(author_label, theme, texts, tool_sentiment_note="", retry=3):
    prompt = build_summary_prompt(author_label, theme, texts, tool_sentiment_note)
    for attempt in range(retry):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=2000,
                ),
            )
            parsed = json.loads(resp.text.strip())
            if parsed.get("sentiment") not in SENTIMENT_OPTIONS:
                parsed["sentiment"] = "kontroversi"
            return parsed
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"[WARN] gagal summarize {author_label} (percobaan {attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    return {"summary": "(gagal diringkas otomatis)", "sentiment": "kontroversi", "reason": "error API"}

print("Fungsi summarization siap.")

### 9b. Jalankan summarization untuk semua top author

In [ ]:
pd.set_option("display.max_colwidth", None)   # jangan potong isi kolom teks panjang
pd.set_option("display.max_rows", None)       # tampilkan semua baris (opsional, hati-hati kalau datanya banyak)

results = []

for _, row in df_top_authors.iterrows():
    author_id = row["author_id"]
    theme = row["theme"]

    author_posts_df = df_topic[(df_topic[COL_AUTHOR_ID] == author_id) & (df_topic["theme"] == theme)]
    posts = author_posts_df["text_clean"].dropna().tolist()

    tool_sentiment_note = ""
    if COL_SENTIMENT in author_posts_df.columns:
        counts = author_posts_df[COL_SENTIMENT].dropna().value_counts()
        if not counts.empty:
            total = counts.sum()
            tool_sentiment_note = ", ".join(f"{label} {round(100 * n / total)}%" for label, n in counts.items())

    posts = posts[:MAX_POSTS_PER_AUTHOR_SUMMARY]
    if not posts:
        continue

    print(f"Meringkas @{author_id} | tema={theme} | {len(posts)} post ...")
    result = summarize_author(author_id, theme, posts, tool_sentiment_note)

    results.append({
        "theme": theme,
        "rank": int(row["rank"]),
        "author_id": author_id,
        "score": round(row["score"], 2),
        "summary": result.get("summary", ""),
        "sentiment": result.get("sentiment", "kontroversi"),
        "reason": result.get("reason", ""),
    })

df_summary = pd.DataFrame(results)

print("\n=== Hasil akhir: ringkasan & sentimen per top author ===")
display(df_summary)

# --- Export ke Excel ---
file_name = "hasil_analisis_top_author.xlsx"
df_summary.to_excel(file_name, index=False)
print(f"\n[OK] Data berhasil diekspor ke {file_name}")

from google.colab import files
files.download(file_name)